In [11]:
# import necessary packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
sns.set(rc={'axes.facecolor': 'grey'})
plt.rcParams['figure.dpi'] = 300

In [12]:
import os
os.chdir('C:/Users/edwin/OneDrive/Documents/GitHub/CHC')

In [13]:
# define tercile cutoff helper functions

# get the tercile cutoffs of a dataframe
def get_tercile_cutoffs(df):
    return df.quantile([0.33, 0.66])

# Assign the Tercile Category for both predicted_precip and precip
def assign_tercile_category(value, lower_cutoff, upper_cutoff):
    if value <= lower_cutoff:
        return 'Low'
    elif value <= upper_cutoff:
        return 'Medium'
    else:
        return 'High'

In [14]:
def compute_tercile_seasonal_df(seasonal_file_path):
    """Takes a file path to seasonal csv file and computes tercile
    agreement for each month of prediction and season.

    Requires helper functions get_tercile_cutoffs and assign_tercile_category,
    which are defined above.

    Arguments
    ---------
    seasonal_file_path (str): file path to seasonal csv file
    make sure that the path provided is a path to the actual file:
    /data/csv/{some_region}_{some_model}_merged_seasonal.csv

    The csv file has the following columns:
    'date_of_prediction' - model's date of prediction as YYYY-MM-DD
    'season' - season that the model is predicting, i.e. JJA, JAS, SON, DJF, MAM
    'predicted_precip' - predicted precipitation from model
    'precip' - actual precipitation from CHIRPS
    'model' - model name, i.e. NCEP, CFSv2, ECMWF, GFS
    'region' - region name, i.e. eastern_east_africa, west_africa, etc.

    Returns
    -------
    dataframe with tercile agreement for each month of prediction and season

    The resulting dataframe has the following columns:
    'year_of_prediction' - year of prediction as YYYY
    'season' - season that the model is predicting, i.e. JJA, JAS, SON, DJF, MAM
    'month_of_prediction' - month of prediction as MM
    'predicted_precip' - predicted precipitation from model
    'precip' - actual precipitation from CHIRPS
    f'{model}_tercile_class' - tercile class (low/medium/high) of predicted precipitation from model,
                                column name changes dynamically
    'chirps_tercile_class' - tercile class of actual precipitation from CHIRPS
    'agreement' - 1 if terciles agree, 0 otherwise
    'model' - model name, i.e. NCEP, CFSv2, ECMWF, GFS
    'region' - region name, i.e. eastern_east_africa, west_africa, etc.
    """

    # open seasonal file
    df = pd.read_csv(seasonal_file_path, index_col=0)

    # Convert 'date_of_prediction' to datetime
    df['date_of_prediction'] = pd.to_datetime(df['date_of_prediction'])
    df['month_of_prediction'] = df['date_of_prediction'].dt.month
    df['year_of_prediction'] = df['date_of_prediction'].dt.year



    # Iterate over every unique combination of month_of_prediction and season over all years
    for (month, season), group in df.groupby(['month_of_prediction', 'season']):

        model = df['model'].iloc[0]
        region = df['region'].iloc[0]

        cutoffs = get_tercile_cutoffs(group[['predicted_precip', 'precip']])

        # extract cutoffs for predicted precip and precip
        lower_cutoff_predicted = cutoffs.loc[0.33, 'predicted_precip']
        upper_cutoff_predicted = cutoffs.loc[0.66, 'predicted_precip']

        lower_cutoff_precip = cutoffs.loc[0.33, 'precip']
        upper_cutoff_precip = cutoffs.loc[0.66, 'precip']

        # assign tercile categories for predicted_precip and precip
        group[f'{model}_tercile_class'] = group['predicted_precip'].apply(assign_tercile_category, args=(lower_cutoff_predicted, upper_cutoff_predicted))
        group['chirps_tercile_class'] = group['precip'].apply(assign_tercile_category, args=(lower_cutoff_precip, upper_cutoff_precip))

        # calculate agreement, 1 if terciles agree, 0 otherwise
        group['agreement'] = (group[f'{model}_tercile_class'] == group['chirps_tercile_class']).astype(int)

        # Update the DataFrame with the new columns
        df.loc[group.index, f'{model}_tercile_class'] = group[f'{model}_tercile_class']
        df.loc[group.index, 'chirps_tercile_class'] = group['chirps_tercile_class']
        df.loc[group.index, 'agreement'] = group['agreement']
        df.loc[group.index, 'model'] = model
        df.loc[group.index, 'region'] = region

    # Select the desired columns for the final DataFrame
    final_columns = ['year_of_prediction', 'season', 'month_of_prediction', 'predicted_precip', 'precip', f'{model}_tercile_class', 'chirps_tercile_class', 'agreement', 'model', 'region']
    final_df = df[final_columns]

    return final_df


In [15]:
# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

# initiate file list
list_of_files = glob.glob('data/csv/*.csv')

# Loop over all files
for f in list_of_files:
    # Generate the DataFrame
    df = compute_tercile_seasonal_df(f)

    # Store the DataFrame in the dictionary with the year as key
    dfs_dict[f] = df

# Concatenate all DataFrames in the dictionary into one DataFrame
final_df = pd.concat(dfs_dict.values(), ignore_index=True)

# Now final_df contains all concatenated data

In [16]:
# subset the final_df by tercile class, low, medium, and high for plotting

# subset by chirps tercile class = low
region_model_df_low = final_df.query('chirps_tercile_class == "Low"')

# compute agreement rates
region_model_df_low = region_model_df_low.groupby(['month_of_prediction', 'season', 'model', 'region'])[['agreement']].mean().reset_index()

region_model_df_low.to_csv('data/csv/metrics/region_model_df_low.csv')

# subset by chirps tercile class = medium
region_model_df_medium = final_df.query('chirps_tercile_class == "Medium"')

# compute agreement rates
region_model_df_medium = region_model_df_medium.groupby(['month_of_prediction', 'season', 'model', 'region'])[['agreement']].mean().reset_index()

# subset by chirps tercile class = High
region_model_df_high = final_df.query('chirps_tercile_class == "High"')

# compute agreement rates
region_model_df_high = region_model_df_high.groupby(['month_of_prediction', 'season', 'model', 'region'])[['agreement']].mean().reset_index()

region_model_df_high.to_csv('data/csv/metrics/region_model_df_high.csv')

In [ ]:
region_model_df_high


,month_of_prediction,season,model,region,agreement
0,1,AMJ,CCSM4,eastern_ukraine,0.454545
1,1,AMJ,CESM1,eastern_ukraine,0.400000
2,1,AMJ,CMCC,eastern_ukraine,0.500000
3,1,AMJ,CanESM5,eastern_ukraine,0.300000
4,1,AMJ,DWD,eastern_ukraine,0.272727
...,...,...,...,...,...
1299,12,MJJ,JMA,south_sudan,0.200000
1300,12,MJJ,METEO,south_sudan,0.400000
1301,12,MJJ,MME,south_sudan,0.454545
1302,12,MJJ,NASA,south_sudan,0.363636


In [9]:
region_model_df_low

,month_of_prediction,season,model,region,agreement
0,1,AMJ,CCSM4,eastern_ukraine,0.363636
1,1,AMJ,CESM1,eastern_ukraine,0.400000
2,1,AMJ,CMCC,eastern_ukraine,0.333333
3,1,AMJ,CanESM5,eastern_ukraine,0.444444
4,1,AMJ,DWD,eastern_ukraine,0.400000
...,...,...,...,...,...
1285,12,MJJ,JMA,south_sudan,0.400000
1286,12,MJJ,METEO,south_sudan,0.400000
1287,12,MJJ,MME,south_sudan,0.545455
1288,12,MJJ,NASA,south_sudan,0.272727


In [10]:
region_model_df_medium

,month_of_prediction,season,model,region,agreement
0,1,AMJ,CCSM4,eastern_ukraine,0.100000
1,1,AMJ,CESM1,eastern_ukraine,0.300000
2,1,AMJ,CMCC,eastern_ukraine,0.333333
3,1,AMJ,CanESM5,eastern_ukraine,0.000000
4,1,AMJ,DWD,eastern_ukraine,0.400000
...,...,...,...,...,...
1285,12,MJJ,JMA,south_sudan,0.200000
1286,12,MJJ,METEO,south_sudan,0.333333
1287,12,MJJ,MME,south_sudan,0.400000
1288,12,MJJ,NASA,south_sudan,0.300000


In [11]:
# for predictions made in January for the AMJ season in eastern ukraine, CESM1 agrees with chirps 40% of the time for the upper tercile

In [17]:
# Create a combined column for the region-season pair
region_model_df_high['region_season'] = region_model_df_high['region'] + " | " + region_model_df_high['season']

# Sort the DataFrame by region (and season if needed) to ensure proper ordering of facets
df_sorted = region_model_df_high.sort_values(by=['region', 'season'])
facet_order = df_sorted['region_season'].unique().tolist()

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='agreement')
    
    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }
    
    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]
    
    # Create the heatmap and force y ticklabels to remain visible
    ax = sns.heatmap(
        d,
        vmin=0, vmax=1,
        cmap=sns.color_palette('RdYlGn', 10),
        fmt=".2f",
        linewidths=0.1,
        linecolor='black',
        square=True,
        yticklabels=True  # ensure ticklabels are drawn
    )
    
    # Set tick label properties explicitly
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=7)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)

    ax.invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(region_model_df_high, col='region_season', sharex=False, sharey=False, col_wrap=4, col_order=facet_order)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('AN Tercile Agreement Rate by Model and Month of Prediction', y=0.98)

fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")
plt.savefig('figures/seasonal_metrics/AN_tercile_seasonal.png')
plt.close()

In [18]:
# Create a combined column for the region-season pair
region_model_df_low['region_season'] = region_model_df_low['region'] + " | " + region_model_df_low['season']

# Sort the DataFrame by region (and season if needed) to ensure proper ordering of facets
df_sorted = region_model_df_low.sort_values(by=['region', 'season'])
facet_order = df_sorted['region_season'].unique().tolist()

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='agreement')

    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }

    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]

    # Create the heatmap and force y ticklabels to remain visible
    ax = sns.heatmap(
        d,
        vmin=0, vmax=1,
        cmap=sns.color_palette('RdYlGn', 10),
        fmt=".2f",
        linewidths=0.1,
        linecolor='black',
        square=True,
        yticklabels=True  # ensure ticklabels are drawn
    )
    
    # Set tick label properties explicitly
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=7)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)

    ax.invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(region_model_df_low, col='region_season', sharex=False, sharey=False, col_wrap=4, col_order=facet_order)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('BN Tercile Agreement Rate by Model and Month of Prediction', y=0.98)

fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")
plt.savefig('figures/seasonal_metrics/BN_tercile_seasonal.png')
plt.close()

In [8]:
# Create a combined column for the region-season pair
region_model_df_medium['region_season'] = region_model_df_medium['region'] + " | " + region_model_df_medium['season']

# Sort the DataFrame by region (and season if needed) to ensure proper ordering of facets
df_sorted = region_model_df_medium.sort_values(by=['region', 'season'])
facet_order = df_sorted['region_season'].unique().tolist()

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='agreement')

    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }

    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]

    sns.heatmap(d,
                vmin=0, vmax=1,
                cmap=sns.color_palette('RdYlGn', 10),
                fmt=".2f",
                linewidths=0.1, linecolor='black',
                square=True)
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(region_model_df_medium, col='region_season', sharex=False, sharey=False, col_wrap=4, col_order=facet_order)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('BN Tercile Agreement Rate by Model and Month of Prediction', y=0.98)

fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")
plt.savefig('figures/seasonal_metrics/N_tercile_seasonal.png')
plt.close()